# ML Zoomcamp 2026 - Homework 3
Machine Learning for Classification (pinned 2026 lead-scoring release)

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

## Data preparation

In [2]:
!wget -q -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

df = pd.read_csv('course_lead_scoring_2026.csv')
df.shape  # (5000, 9)

(5000, 9)

In [3]:
df.isnull().sum()
# lead_source       148   industry      240   employment_status 194
# location          208   annual_income 369   lead_score         35

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
categorical = list(df.dtypes[df.dtypes == 'object'].index)
numerical = [c for c in df.columns if c not in categorical + ['converted']]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

categorical, numerical

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'])

## Q1. Mode for `industry`

In [5]:
df.industry.mode()[0]       # 'technology'
df.industry.value_counts()  # technology 1173, retail 925, healthcare 840, ...

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

## Q2. Biggest correlation

In [6]:
corr = df[numerical].corr()
corr.round(4)

# interaction_count        & lead_score        -> 0.9157  <-- biggest
# number_of_courses_viewed & lead_score        -> 0.7572
# number_of_courses_viewed & interaction_count -> 0.7216
# annual_income            & interaction_count -> 0.1228

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.0000,0.1613,0.1228,0.2295
number_of_courses_viewed,0.1613,1.0000,0.7216,0.7572
interaction_count,0.1228,0.7216,1.0000,0.9157
lead_score,0.2295,0.7572,0.9157,1.0000


## Split the data (60% / 20% / 20%)

In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)  # 3000, 1000, 1000

(3000, 1000, 1000)

## Q3. Mutual information (training set only)

In [8]:
for c in categorical:
    print(c, round(mutual_info_score(df_train[c], y_train), 2))

# lead_source        0.03   <-- biggest
# employment_status  0.02
# industry           0.00
# location           0.00

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


## Q4. Logistic regression with one-hot encoding

In [9]:
features = categorical + numerical

def train_and_score(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    X_val = dv.transform(df_val[feats].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return accuracy_score(y_val, model.predict(X_val))


accuracy = train_and_score(features)
accuracy, round(accuracy, 2)   # 0.645 -> 0.65

(0.645, 0.65)

## Q5. Feature elimination

In [10]:
differences = {}
for f in features:
    acc_without = train_and_score([x for x in features if x != f])
    differences[f] = accuracy - acc_without
    print(f'{f:26s} acc_without={acc_without:.3f}  diff={accuracy - acc_without:+.3f}')

# lead_source               acc_without=0.642  diff=+0.003
# industry                  acc_without=0.645  diff=+0.000
# employment_status         acc_without=0.645  diff=+0.000
# location                  acc_without=0.645  diff=+0.000
# annual_income             acc_without=0.724  diff=-0.079   (removing it HELPS a lot)
# number_of_courses_viewed  acc_without=0.643  diff=+0.002   <-- smallest of the 3 options
# interaction_count         acc_without=0.601  diff=+0.044
# lead_score                acc_without=0.644  diff=+0.001

lead_source                acc_without=0.642  diff=+0.003
industry                   acc_without=0.645  diff=+0.000
employment_status          acc_without=0.645  diff=+0.000
location                   acc_without=0.645  diff=+0.000
annual_income              acc_without=0.724  diff=-0.079
number_of_courses_viewed   acc_without=0.643  diff=+0.002
interaction_count          acc_without=0.601  diff=+0.044
lead_score                 acc_without=0.644  diff=+0.001


In [11]:
# among the three listed options
{f: round(differences[f], 3) for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']}
# -> number_of_courses_viewed (0.002) is the smallest

{'lead_source': 0.003,
 'number_of_courses_viewed': 0.002,
 'interaction_count': 0.044}

## Q6. Regularization

In [12]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C), 3))

# 1e-06  0.598
# 1e-05  0.598
# 1e-04  0.613
# 1e-03  0.645   <-- best

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


## Answers
| Q | Question | Answer |
|---|---|---|
| 1 | Mode for `industry` | `technology` (1173) |
| 2 | Biggest correlation | `interaction_count` and `lead_score` (0.9157) |
| 3 | Biggest mutual information | `lead_source` (0.03) |
| 4 | Validation accuracy | 0.65 (raw 0.645) |
| 5 | Smallest difference | `number_of_courses_viewed` (+0.002) |
| 6 | Best `C` | 0.001 (accuracy 0.645) |

**Side observation (not graded):** dropping `annual_income` *raises* validation accuracy from
0.645 to 0.724. It is an unscaled feature with a huge range, so with `C=1.0` and no scaling it
dominates the linear combination and hurts the model. A reminder that logistic regression wants
standardized numeric features.